# 11 - Online Monitoring, Promotion, and Rollback (SmartQueue)

In this tutorial, we run a full online monitoring experiment for the SmartQueue serving role.

The objective is to demonstrate that the serving stack can:

- monitor operational behavior of the deployed model service,
- monitor model outputs over time,
- monitor a user-feedback proxy,
- and apply explicit promotion/rollback triggers.

This notebook is structured as a **lab runbook** you can follow step-by-step during the experiment.

## Experiment resources

For this experiment, we provision one VM on `KVM@TACC` and run:

- SmartQueue serving API (FastAPI)
- Navidrome (integration path)
- Prometheus (metrics collection)
- Grafana (dashboards + alerts)
- optional cAdvisor (container-level metrics)

Prerequisites:

- Chameleon account + project
- SSH key uploaded to KVM@TACC
- access to MLflow tracking URI and model registry

In [ ]:
## 1) Launch and set up a VM (runs in Chameleon Jupyter environment)

Run the following cells in Chameleon Jupyter.

# runs in Chameleon Jupyter environment
from chi import server, context, lease, network
import chi, os, datetime

context.version = "1.0"
context.choose_project()
context.choose_site(default="KVM@TACC")

username = os.getenv("USER")
LEASE_NAME = f"lease-smartqueue-monitor-{username}"
SERVER_NAME = f"node-smartqueue-monitor-{username}"
FLAVOR_NAME = "m1.medium"
IMAGE_NAME = "CC-Ubuntu24.04"

In [ ]:
# runs in Chameleon Jupyter environment
l = lease.Lease(LEASE_NAME, duration=datetime.timedelta(hours=6))
l.add_flavor_reservation(id=chi.server.get_flavor_id(FLAVOR_NAME), amount=1)
l.submit(idempotent=True)
l.show()

# runs in Chameleon Jupyter environment
s = server.Server(
    SERVER_NAME,
    image_name=IMAGE_NAME,
    flavor_name=l.get_reserved_flavors()[0].name,
)
s.submit(idempotent=True)
s.associate_floating_ip()
s.refresh()
s.check_connectivity()
s.show(type="widget")

In [ ]:
# runs in Chameleon Jupyter environment
security_groups = [
    {"name": "allow-ssh", "port": 22, "description": "Enable SSH"},
    {"name": "allow-4533", "port": 4533, "description": "Navidrome"},
    {"name": "allow-8000", "port": 8000, "description": "Serving API"},
    {"name": "allow-9090", "port": 9090, "description": "Prometheus"},
    {"name": "allow-3000", "port": 3000, "description": "Grafana"},
    {"name": "allow-8080", "port": 8080, "description": "cAdvisor"},
]

for sg in security_groups:
    secgroup = network.SecurityGroup({"name": sg["name"], "description": sg["description"]})
    secgroup.add_rule(direction="ingress", protocol="tcp", port=sg["port"])
    secgroup.submit(idempotent=True)
    s.add_security_group(sg["name"])

print(f"updated security groups: {[sg['name'] for sg in security_groups]}")

In [ ]:
## 2) Prepare server software (runs in Chameleon Jupyter environment)

Install Docker and clone repos:

# runs in Chameleon Jupyter environment
s.execute("curl -sSL https://get.docker.com/ | sudo sh")
s.execute("sudo groupadd -f docker; sudo usermod -aG docker $USER")
s.execute("sudo apt-get update && sudo apt-get install -y docker-compose-plugin git")
s.execute("git clone https://github.com/yanghao13111/SmartQueue.git ~/smartqueue")

In [ ]:
## 3) SSH into the VM (runs on your local terminal)

```bash
ssh -i ~/.ssh/id_rsa_chameleon cc@A.B.C.D
```

Replace `A.B.C.D` with the floating IP shown in `s.show(type="widget")`.

In [ ]:
## 4) Run SmartQueue serving + Navidrome (runs on VM shell)

### 4.1 Start SmartQueue serving

```bash
cd ~/smartqueue/serving/docker
MLFLOW_TRACKING_URI=<YOUR_MLFLOW_URL> \
MODEL_NAME=smartqueue-ranking \
MODEL_STAGE=Production \
UVICORN_WORKERS=1 \
docker compose -f docker-compose-lightgbm.yaml up -d --build --force-recreate fastapi_lgbm
```

Verify:

```bash
curl http://localhost:8000/health
```

### 4.2 Start Navidrome with SmartQueue integration

Set env vars:

```bash
export ND_SMARTQUEUE_ENABLED=true
export ND_SMARTQUEUE_SERVICEURL=http://127.0.0.1:8000
export ND_SMARTQUEUE_TIMEOUT=30s
```

Then run Navidrome using your normal startup method.

## 5) Add monitoring stack (runs on VM shell)

Create a monitoring compose file (Prometheus + Grafana) or use your existing one.

Start monitoring services:

```bash
docker compose -f docker-compose-monitoring.yaml up -d
```

Check endpoints:

- Prometheus: `http://A.B.C.D:9090`
- Grafana: `http://A.B.C.D:3000`
- Serving metrics: `http://A.B.C.D:8000/metrics` (if exposed)

If `/metrics` is missing, add FastAPI instrumentation and redeploy before continuing.

In [ ]:
## 6) Generate load and collect monitoring evidence

Use any client (notebook script, curl loop, or locust) to generate traffic.

Minimum scenarios to run:

1. Normal load (success-dominant)
2. Bursty load (concurrency ramp up/down)
3. Error-inducing load (bad payload mix)

Required screenshots:

- service latency panel (p50/p95/p99)
- request rate and error rate panel
- model output distribution panel
- alert state transitions (`Normal` -> `Pending` -> `Firing`)


## 7) Promotion/rollback trigger policy (use this exact section in report)

### Promote to Production only if all are true for a 30-minute canary window:

- health endpoint stable (no consecutive failures),
- error rate <= 1%,
- p95 latency <= 800 ms,
- invalid score rate (outside [0,1]) <= 0.1%,
- feedback-proxy KPI is not worse than baseline by more than 5%.

### Roll back immediately if any are true:

- 3 consecutive health-check failures,
- error rate > 2% for 5+ minutes,
- p95 latency > 1200 ms for 10+ minutes,
- invalid score rate > 0.5%,
- feedback-proxy KPI drops > 10% relative to baseline.

### Hold (do not promote, do not rollback) if between thresholds.


In [ ]:
## 8) Verify model-output monitoring

Track at least:

- average engagement score over time,
- score histogram buckets (0-0.2, 0.2-0.4, ...),
- invalid score count.

Acceptance criterion:

- no out-of-range scores,
- no abrupt unexplained distribution shift during stable traffic.

## 9) Verify user-feedback proxy monitoring

Use one proxy KPI (based on available data):

- skip-rate after recommendation, or
- completion/watch-time ratio, or
- session abandonment rate.

Document baseline value and candidate value in your report.
Apply the trigger thresholds from section 7.

In [ ]:
## 10) Submission evidence checklist

- [ ] Screenshot: operational dashboard (latency/error/throughput)
- [ ] Screenshot: model-output dashboard (score distribution)
- [ ] Screenshot: feedback-proxy metric trend
- [ ] Screenshot: alert transitions (`Normal/Pending/Firing`)
- [ ] Written trigger policy included in report
- [ ] Short explanation of one promote/no-promote decision from observed metrics

## 11) Teardown (runs in Chameleon Jupyter environment)

In [ ]:
# runs in Chameleon Jupyter environment
# Use only when fully finished with the experiment.
from chi import server, lease
import os

username = os.getenv("USER")
s = server.get_server(f"node-smartqueue-monitor-{username}")
l = lease.get_lease(f"lease-smartqueue-monitor-{username}")

s.delete()
l.delete()